# Gaming Toxicity Detection — Granularity Experiment

**Research question:** At what number of classes does toxicity classification performance peak before declining?

**What this notebook does:**
1. Class balance check at every granularity step
2. Per-dataset incremental experiment (WoT 2→6 classes, Dota 2→4 classes) — in-game F1 only
3. Sweet spot line plot
4. Centroid clustering analysis — gateway decision for unified cross-game label scheme

## 0. Imports & Configuration

In [ ]:
# Standard imports — keep consistent with binary experiment style
import warnings, time, json
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from imblearn.over_sampling import RandomOverSampler
import sys
sys.path.insert(0, str(Path('..').resolve()))
from src.loaders import load_wot, load_dota
from src.pipelines import build_pipe
from src.scoring import append_registry
from src.label_schemes import WOT_SCHEMES, DOTA_SCHEMES, WOT_CLASS_NAMES, DOTA_CLASS_NAMES

CONFIG = {
    'seed': 7524,
    'cv_folds': 5,
    'text_col': 'clean_message',
    'label_col': 'label',
    'registry_path': Path('../data/results/results_registry.csv'),
    'min_class_size': 500,
}
seed = CONFIG['seed']
cv   = StratifiedKFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=seed)
np.random.seed(seed)
print('CONFIG loaded.')

All experiments in this notebook write to the shared results registry at
`data/results/results_registry.csv`. The dashboard notebook (05) reads from
this registry to produce all comparison plots.

## 1. Class Balance Check

In [ ]:
# Verify class counts at every granularity step before training.
# Any class < 500 samples is too small for 5-fold CV — flag it.
MINIMUM_CLASS_SIZE = CONFIG['min_class_size']

print('=== WoT class balance per granularity ===')
for n in [2, 3, 4, 5, 6]:
    df = load_wot('train', scheme=WOT_SCHEMES[n])
    counts = df[CONFIG['label_col']].value_counts().sort_index()
    small = counts[counts < MINIMUM_CLASS_SIZE]
    flag = f'  ⚠ classes too small: {small.to_dict()}' if len(small) else ''
    print(f'  n={n}: {counts.to_dict()}{flag}')

print('\n=== Dota class balance per granularity ===')
for n in [2, 3, 4]:
    df = load_dota('train', scheme=DOTA_SCHEMES[n])
    counts = df[CONFIG['label_col']].value_counts().sort_index()
    small = counts[counts < MINIMUM_CLASS_SIZE]
    flag = f'  ⚠ classes too small: {small.to_dict()}' if len(small) else ''
    print(f'  n={n}: {counts.to_dict()}{flag}')

If any class has fewer than 500 samples, that granularity step is skipped —
RandomOverSampler handles imbalance within folds but cannot create meaningful
signal from very few genuine samples.

## 2. Per-Dataset Incremental Experiment

In [ ]:
# Run all 3 models at every granularity level for each game.
# Best model may change at higher n_classes — do NOT inherit binary winner.

def run_granularity_track(game: str, schemes: dict, class_names_map: dict,
                           load_fn, registry_path: Path):
    """For each n_classes: run LR, NB, SVC, record best in registry."""
    results = []

    for n_classes, scheme in schemes.items():
        train_df = load_fn('train', scheme=scheme)
        X_train  = train_df[CONFIG['text_col']]
        y_train  = train_df[CONFIG['label_col']]

        # Skip if any class < MINIMUM_CLASS_SIZE
        counts = y_train.value_counts()
        if counts.min() < MINIMUM_CLASS_SIZE:
            print(f'  {game} n={n_classes}: skipped (min class size: {counts.min()})')
            continue

        classifiers = {
            'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000,
                                                       random_state=seed, n_jobs=1),
            'Naive Bayes':         MultinomialNB(),
            'LinearSVC':           LinearSVC(C=1.0, max_iter=2000, tol=1e-3,
                                             random_state=seed),
        }

        best_f1, best_name = -1, None
        for clf_name, clf in classifiers.items():
            pipe   = build_pipe(clone(clf), oversampler=RandomOverSampler(random_state=seed))
            scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                                      scoring='f1_macro', n_jobs=-1)
            cv_f1  = scores.mean()
            cv_std = scores.std()
            print(f'  {game} n={n_classes} {clf_name:<22} cv_macro_f1={cv_f1:.4f} ± {cv_std:.4f}')

            if cv_f1 > best_f1:
                best_f1, best_name = cv_f1, clf_name

            append_registry({
                'experiment':       'granularity_per_dataset',
                'train_game':       game,
                'test_game':        game,
                'n_classes':        n_classes,
                'label_scheme':     'native',
                'model':            clf_name,
                'cv_macro_f1':      round(cv_f1, 4),
                'cv_std':           round(cv_std, 4),
                'cv_weighted_f1':   None,
                'test_macro_f1':    None,
                'test_weighted_f1': None,
                'per_class_recall': None,
                'ood_macro_f1':     None,
                'ood_weighted_f1':  None,
                'anomaly_auroc':    None,
                'notes':            '',
            }, path=registry_path)

        results.append({'game': game, 'n_classes': n_classes,
                        'best_model': best_name, 'best_cv_f1': round(best_f1, 4)})
        print(f'  → best at n={n_classes}: {best_name} ({best_f1:.4f})\n')

    return pd.DataFrame(results)

print('--- WoT incremental ---')
wot_results = run_granularity_track('WoT', WOT_SCHEMES, WOT_CLASS_NAMES,
                                     load_wot, CONFIG['registry_path'])
print('\n--- Dota incremental ---')
dota_results = run_granularity_track('Dota', DOTA_SCHEMES, DOTA_CLASS_NAMES,
                                      load_dota, CONFIG['registry_path'])

print('\nWoT summary:')
print(wot_results.to_string(index=False))
print('\nDota summary:')
print(dota_results.to_string(index=False))

## 2. Per-Dataset Incremental Experiment — Results

Each model is evaluated at every granularity level independently — we do not
assume the binary winner stays best at 6 classes. The sweet spot is the
`n_classes` where `cv_macro_f1` peaks before declining.

OOD evaluation is not possible here because WoT and Dota have different native
label spaces at n > 2. Binary OOD is already established from the binary
experiment notebook.

## 3. Sweet Spot Plot

In [ ]:
# Visualise F1 vs n_classes to identify the inflection point.
# This is the core finding of this notebook.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (game, df) in zip(axes, [('WoT', wot_results), ('Dota', dota_results)]):
    if df.empty:
        ax.set_title(f'{game} — no results')
        continue
    ax.plot(df['n_classes'], df['best_cv_f1'], marker='o', linewidth=2, color='#1565C0')
    for _, row in df.iterrows():
        ax.annotate(f"{row['best_cv_f1']:.3f}\n({row['best_model'][:3]})",
                    (row['n_classes'], row['best_cv_f1']),
                    textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
    ax.set_xlabel('Number of Classes', fontsize=12)
    ax.set_ylabel('CV Macro F1', fontsize=12)
    ax.set_title(f'{game} — F1 vs Granularity', fontweight='bold')
    ax.set_xticks(df['n_classes'])
    ax.set_ylim(0.5, 1.0)
    ax.grid(True, alpha=0.3)

plt.suptitle('Granularity Sweet Spot — Both Games', fontweight='bold', fontsize=14)
plt.tight_layout()
Path('../data/results').mkdir(parents=True, exist_ok=True)
plt.savefig('../data/results/granularity_sweet_spot.png', dpi=150, bbox_inches='tight')
plt.show()

The line plot reveals the `n_classes` where F1 peaks. The annotated model name
(first 3 chars) shows whether the best model changes across granularities.
Declining F1 after the peak = classes are too similar for TF-IDF features to
discriminate — this is the answer to the core research question.

## 4. Centroid Clustering — Unified Scheme Gateway

In [ ]:
# Cosine similarity between WoT and Dota class centroids.
# If cross-game clusters are interpretable (sim > 0.3), define unified scheme.
# Otherwise OOD stays at binary only.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

wot_train  = load_wot('train')
dota_train = load_dota('train')

# Fit joint TF-IDF on combined corpus for shared vocabulary
combined_texts = pd.concat([wot_train[CONFIG['text_col']],
                              dota_train[CONFIG['text_col']]])
joint_tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=5, max_df=0.95,
                                sublinear_tf=True, norm='l2')
joint_tfidf.fit(combined_texts)

X_wot_joint  = joint_tfidf.transform(wot_train[CONFIG['text_col']])
X_dota_joint = joint_tfidf.transform(dota_train[CONFIG['text_col']])

# Compute per-class centroids in joint vocab space
wot_centroids  = {c: np.asarray(X_wot_joint[(wot_train[CONFIG['label_col']].astype(int)==c).values].mean(axis=0))
                  for c in range(6)}
dota_centroids = {c: np.asarray(X_dota_joint[(dota_train[CONFIG['label_col']].astype(int)==c).values].mean(axis=0))
                  for c in range(4)}

wot_mat  = normalize(np.vstack([wot_centroids[c]  for c in range(6)]))
dota_mat = normalize(np.vstack([dota_centroids[c] for c in range(4)]))
sim_matrix = cosine_similarity(wot_mat, dota_mat)

wot_labels_display  = ['WoT: ' + n for n in WOT_CLASS_NAMES[6]]
dota_labels_display = ['Dota: ' + n for n in DOTA_CLASS_NAMES[4]]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(sim_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=dota_labels_display, yticklabels=wot_labels_display, ax=ax)
ax.set_title('WoT × Dota Class Centroid Cosine Similarity', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../data/results/centroid_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nMax similarity per WoT class:')
for i, wot_name in enumerate(WOT_CLASS_NAMES[6]):
    j = int(np.argmax(sim_matrix[i]))
    print(f'  WoT {wot_name:<18} ↔ Dota {DOTA_CLASS_NAMES[4][j]:<12} sim={sim_matrix[i,j]:.3f}')

print('\nDecision rule: similarity > 0.3 between semantically similar classes → consider unified scheme')

## 4. Centroid Clustering — Decision

**Decision rule:** If the heatmap shows interpretable cross-game clusters
(similarity > 0.3 between semantically similar classes), define a unified
label scheme in `src/label_schemes.py` and run OOD at that granularity.
If not, OOD stays at binary level only — document as limitation.

This avoids imposing a subjective label mapping: the data decides whether
cross-game unification is linguistically justified.